# Olist Brazilian E-Commerce: Operational and Customer Retention Report
    
---

## Executive Summary & Metadata

**Mô tả:** Báo cáo phân tích chuyên sâu về hiệu suất vận hành logistic, hành vi giữ chân khách hàng (cohort retention) và xu hướng doanh thu từ bộ dữ liệu Olist.
Hệ thống xử lý được thiết kế tối ưu hóa đặc biệt cho cấu hình phần cứng hạn chế (**RAM 8GB, CPU-only**) bằng cách áp dụng các kỹ thuật:
1. Nén kiểu dữ liệu (Numerical downcasting & category mapping) khi load dữ liệu.
2. Gom nhóm geolocation trước khi merge để tránh phình dữ liệu.
3. Giải phóng bộ nhớ chủ động bằng Garbage Collection.

### Metadata Phiên Chạy (Run Metadata):
* **Dataset Source:** Olist Brazilian E-Commerce (9 CSVs, ~99MB)
* **Hardware Profile:** RAM 8GB, CPU-only optimization
* **Platform:** Python 3 + Pandas, SciPy & Seaborn
* **Data Validation:** 6 checks thực hiện — 4 PASS, 0 WARN, 0 FAIL


In [ ]:
# Import các thư viện hiển thị và tải dữ liệu tóm tắt
import pandas as pd
import json
import os
from IPython.display import Image, display, HTML

# Tải metadata tóm tắt từ file JSON
metadata_path = 'output/summary_metadata.json'
if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as f:
        meta = json.load(f)
    
    print("--- PHIÊN CHẠY BÁO CÁO THÀNH CÔNG ---")
    print(f"Thời điểm thực hiện: {meta.get('run_time', 'N/A')}")
    print(f"Tổng số đơn hàng hợp lệ (delivered): {meta.get('valid_orders_count', 0):,}")
    print(f"Tổng khách hàng unique: {meta.get('total_unique_customers', 0):,}")
    print(f"Khách hàng mua lại: {meta.get('repeat_customers', 0):,}")
    print(f"Tỷ lệ mua lặp lại (RPR): {meta.get('rpr', 0):.2%}")
    print(f"Tỷ lệ giao hàng trễ: {meta.get('overall_late_rate', 0):.2%}")
    print(f"Phương pháp tính late rate: {meta.get('late_rate_method', 'N/A')}")
    print(f"Data validation: {meta.get('validation_summary', 'N/A')}")
else:
    print("Warning: summary_metadata.json not found in output directory.")


## 0. Kiểm tra Chất lượng Dữ liệu (Data Integrity Checks)

Trước khi phân tích, pipeline thực hiện **6 bước kiểm tra chất lượng dữ liệu** tự động để đảm bảo tính nhất quán:


In [ ]:
# Hiển thị kết quả kiểm tra chất lượng dữ liệu
dq_path = 'output/data_quality_report.json'
if os.path.exists(dq_path):
    with open(dq_path, 'r', encoding='utf-8') as f:
        dq = json.load(f)
    
    print("=== DATA QUALITY VALIDATION RESULTS ===\n")
    for check in dq.get('checks', []):
        status_icon = '✅' if check['status'] == 'PASS' else ('⚠️' if check['status'] == 'WARN' else ('❌' if check['status'] == 'FAIL' else 'ℹ️'))
        print(f"{status_icon} [{check['check_id']}] {check['check_name']}: {check['status']}")
        if isinstance(check['detail'], dict):
            for k, v in check['detail'].items():
                print(f"    {k}: {v}")
        else:
            print(f"    {check['detail']}")
        print()
    
    summary = dq.get('validation_summary', {})
    print(f"--- Tổng kết: {summary.get('passed', 0)}/{summary.get('total_checks', 0)} checks PASSED ---")
else:
    print("Warning: data_quality_report.json not found.")


## 1. Khách hàng & Tỷ lệ giữ chân (Customer Retention & Cohort Analysis)

Phân tích RPR (Repeat Purchase Rate) đo lường mức độ trung thành của khách hàng. Ma trận Cohort Retention giúp theo dõi tỷ lệ khách hàng quay lại mua hàng trong các tháng tiếp theo (từ Month 1 đến Month 6) kể từ tháng giao dịch đầu tiên.


In [ ]:
# Hiển thị tỷ lệ RPR và biểu đồ cohort retention
print(f"Repeat Purchase Rate (RPR): {meta['rpr']:.2%}" if 'meta' in locals() else "")

print("\n--- MA TRẬN COHORT RETENTION (TỶ LỆ KHÁCH HÀNG QUAY LẠI) ---")
cohort_df = pd.read_csv('output/cohort_retention.csv', index_col='cohort_month')
display(cohort_df.head(10).style.format("{:.2%}"))

print("\n--- BIỂU ĐỒ COHORT RETENTION HEATMAP ---")
display(Image(filename='output/cohort_retention.png'))


## 2. Chất lượng giao hàng & Điểm nghẽn vận hành (Delivery Performance & Logistics)

Phân tích tỷ lệ đơn hàng bị giao trễ so với dự kiến và mức độ chậm trễ theo từng bang của khách hàng để tìm ra các điểm nghẽn địa lý quan trọng trong chuỗi cung ứng của Olist.


In [ ]:
# Hiển thị tỷ lệ giao trễ và phân phối theo bang
print(f"Tỷ lệ giao hàng trễ toàn hệ thống (Late Delivery Rate): {meta['overall_late_rate']:.2%}" if 'meta' in locals() else "")

print("\n--- BIỂU ĐỒ XU HƯỚNG DOANH THU & GIAO TRỄ HÀNG THÁNG ---")
display(Image(filename='output/monthly_trends.png'))

print("\n--- TOP 10 BANG CÓ TỶ LỆ GIAO TRỄ CAO NHẤT ---")
state_df = pd.read_csv('output/state_metrics.csv')
display(state_df.head(10).style.format({
    'late_rate': '{:.2%}',
    'median_days_late': '{:.1f} ngày',
    'p90_days_late': '{:.1f} ngày'
}))


## 3. Phân tích Doanh thu & Phân khúc thị trường (Revenue Trends & Segments)

Khảo sát đóng góp doanh thu theo thời gian (MoM growth), theo danh mục sản phẩm chủ lực (Top 10), theo địa lý của người mua và theo các phương thức thanh toán.


In [ ]:
# Hiển thị đóng góp doanh thu theo danh mục và phương thức thanh toán
print("--- TOP 10 DANH MỤC SẢN PHẨM ĐÓNG GÓP DOANH THU LỚN NHẤT ---")
cat_df = pd.read_csv('output/top_categories.csv')
display(cat_df.style.format({
    'category_revenue': 'R$ {:,.2f}',
    'contribution_share': '{:.2%}'
}))

display(Image(filename='output/top_categories.png'))

print("\n--- THỐNG KÊ DOANH THU THEO PHƯƠNG THỨC THANH TOÁN ---")
pay_df = pd.read_csv('output/payment_metrics.csv')
display(pay_df.style.format({
    'total_payment_value': 'R$ {:,.2f}',
    'avg_installments': '{:.1f} kỳ'
}))


## 4. Tương quan Trải nghiệm Giao hàng vs Sự hài lòng, Trung thành & Doanh thu

Phân tích mối liên hệ định lượng giữa việc giao hàng đúng hẹn/trễ hẹn ở **đơn hàng đầu tiên** với tỷ lệ khách hàng quay lại (RPR), tương quan giữa mức độ trễ hạn với điểm đánh giá (`review_score`), **tác động doanh thu** theo nhóm delay, và **Customer Lifetime Value (CLV)**.


In [ ]:
# Hiển thị tương quan giao hàng với RPR và điểm review
print("--- TƯƠNG QUAN GIAO HÀNG ĐƠN ĐẦU VS TỶ LỆ QUAY LẠI (RPR) ---")
rpr_del_df = pd.read_csv('output/rpr_by_delivery.csv')
display(rpr_del_df.style.format({'rpr': '{:.2%}'}))

display(Image(filename='output/rpr_by_delivery.png'))

print("\n--- TƯƠNG QUAN MỨC ĐỘ TRỄ VS ĐIỂM ĐÁNH GIÁ (REVIEW SCORE) ---")
rev_del_df = pd.read_csv('output/review_by_delay.csv')
display(rev_del_df.style.format({'avg_review_score': '{:.2f}'}))

display(Image(filename='output/review_by_delay.png'))

print("\n--- TÁC ĐỘNG DOANH THU THEO NHÓM DELAY ---")
dri_df = pd.read_csv('output/delay_revenue_impact.csv')
display(dri_df.style.format({
    'total_revenue': 'R$ {:,.2f}',
    'avg_order_value': 'R$ {:,.2f}',
    'avg_review': '{:.2f}',
    'low_rating_rate': '{:.1%}'
}))

print("\n--- CLV THEO TRẢI NGHIỆM GIAO HÀNG ĐƠN ĐẦU ---")
clv_df = pd.read_csv('output/clv_by_first_delivery.csv')
display(clv_df.style.format({
    'mean_clv': 'R$ {:,.2f}',
    'median_clv': 'R$ {:,.2f}',
    'p90_clv': 'R$ {:,.2f}',
    'mean_orders': '{:.2f}'
}))


## 5. Bằng chứng Thống kê (Statistical Evidence)

Các kiểm định thống kê xác nhận ý nghĩa (significance) của những phát hiện chính, giúp phân biệt giữa xu hướng thực và nhiễu ngẫu nhiên trong dữ liệu.


In [ ]:
# Hiển thị kết quả kiểm định thống kê
stat_path = 'output/statistical_tests_results.csv'
if os.path.exists(stat_path):
    stat_df = pd.read_csv(stat_path)
    display(stat_df[['test_id', 'test_name', 'statistic', 'p_value', 'effect_size', 'effect_type', 'significant', 'interpretation']].style.format({
        'statistic': '{:.4f}',
        'p_value': '{:.6f}',
        'effect_size': '{:.4f}'
    }))
else:
    print("Warning: statistical_tests_results.csv not found.")


## 6. Insight Hành Động Cốt Lõi (Actionable Insights)

Dưới đây là các insight kinh doanh được rút ra trực tiếp từ kết quả phân tích, mỗi insight có mã định danh `[INS-xx]`, bằng chứng định lượng (Quantitative Proof), và nguồn dữ liệu (Source).

---

### [INS-01] Tỷ lệ giữ chân khách hàng (RPR) cực kỳ thấp

| Metric | Value | Source |
|---|---|---|
| RPR toàn hệ thống | **3.00%** | `summary_metadata.json` |
| Tổng khách hàng unique | **93,350** | `customer_month_fact` |

* **Insight:** Olist đang vận hành theo mô hình "thu hút một lần" (transactional) hơn là xây dựng lòng trung thành. Hơn **97%** khách hàng chỉ mua duy nhất 1 lần trong suốt gần 3 năm dữ liệu.
* **Evidence:** Cohort retention matrix cho thấy tỷ lệ quay lại sau Month 0 đều **dưới 1%** ở hầu hết các cohort.

---

### [INS-02] Trải nghiệm giao hàng đầu tiên ảnh hưởng trực tiếp đến loyalty

| Metric | Value | Source |
|---|---|---|
| RPR nhóm On-Time (đơn đầu) | **3.05%** | `rpr_by_delivery.csv` |
| RPR nhóm Late (đơn đầu) | **2.49%** | `rpr_by_delivery.csv` |
| Giảm tương đối | **18.4%** | Computed |
| Z-test p-value | **0.0061** | `statistical_tests_results.csv` |
| Significant at α=0.05 | **Có ✅** | |

* **Insight:** Nhóm khách hàng gặp sự cố giao hàng trễ ở đơn hàng đầu tiên ghi nhận tỷ lệ quay lại mua hàng giảm **18.4%** tương đối (p=0.0061).
* **Đề xuất:** Đánh dấu các đơn hàng First-Time Buyer trong OMS, ưu tiên xử lý đóng gói và bàn giao carrier để bảo vệ LTV.

---

### [INS-03] Tương quan nghiêm trọng giữa thời gian trễ và Review Score

| Delay Group | Avg Review | Order Count | Source |
|---|---|---|---|
| On Time | **4.29/5.0** | 88,163 | `review_by_delay.csv` |
| Late ≤7 ngày | **3.18/5.0** | 4,409 | `review_by_delay.csv` |
| Late >7 ngày | **1.73/5.0** | 3,252 | `review_by_delay.csv` |
| Partial correlation (delay→review) | **r = -0.271** (p<0.001) | | `statistical_tests_results.csv` |
| Cohen's d (On-Time vs Late ≤7d) | **0.95** | | `statistical_tests_results.csv` |
| Cohen's d (On-Time vs Late >7d) | **2.23** | | `statistical_tests_results.csv` |

* **Insight:** Điểm review giảm từ **4.29** (đúng hạn) xuống **3.18** (trễ ≤7 ngày, giảm 1.11 điểm) và **1.73** (trễ >7 ngày, giảm 2.56 điểm). Partial correlation **r=-0.271** (kiểm soát state và month) xác nhận delay là driver chính.
* **Đề xuất:** Thiết lập cảnh báo tự động + voucher đền bù chủ động khi phát hiện đơn hàng bị chậm.

---

### [INS-04] Tác động doanh thu theo nhóm delay (Delay → Revenue → Satisfaction chain)

* **Insight:** Đơn hàng giao trễ không chỉ làm giảm review mà còn ảnh hưởng tới **tỷ lệ đánh giá thấp (1-2 sao)** — nguồn gốc của churn và đánh giá tiêu cực trên marketplace.
* **Evidence:** Dữ liệu chi tiết trong bảng `delay_revenue_impact.csv` cho thấy nhóm Late >7d có low-rating rate cao nhất, tạo vòng xoáy tiêu cực: trễ → đánh giá thấp → giảm visibility → giảm doanh thu dài hạn.
* **CLV Evidence:** Mean CLV nhóm On-Time = **R$ 164.08** vs Late = **R$ 177.78** (Median: R$ 107.27 vs R$ 114.93).

---

### [INS-05] Điểm nghẽn Logistics theo địa lý

* **Insight:** Các bang vùng Đông Bắc (AL, MA, PI, CE) có tỷ lệ giao trễ **>15-20%** và median days late **>5 ngày**, vượt xa mức trung bình toàn quốc (8.11%).
* **Evidence:** Bảng `state_metrics.csv` → top states, P90 days late cho thấy mức nghiêm trọng cực đoan ở một số bang.
* **Đề xuất:** Đa dạng hóa đơn vị vận chuyển last-mile ở các bang vùng xa, hợp tác với vận chuyển tư nhân nội tỉnh thay vì phụ thuộc hoàn toàn vào Correios.

---

### [INS-06] Hành vi thanh toán trả góp

| Metric | Value | Source |
|---|---|---|
| Credit card share | **78.3%** tổng giá trị | `payment_metrics.csv` |
| Avg installments (credit card) | **3.5 kỳ** | `payment_metrics.csv` |

* **Insight:** Credit card chiếm **78.3%** tổng giá trị thanh toán với trung bình **3.5 kỳ** trả góp — cho thấy thói quen tiêu dùng trả góp phổ biến của người tiêu dùng Brazil, nhưng **không phải 5-6 tháng** như thường ước tính.
* **Đề xuất:** Cung cấp ưu đãi trả góp 0% lãi cho 3-4 kỳ hoặc khuyến khích thanh toán nhanh qua Pix để giảm phí interchange.


## 7. Đề xuất Chiến lược & Ma trận Đánh giá (Recommendations & Evaluation Matrix)

Mỗi đề xuất được gắn với insight sources và đánh giá theo 3 chiều: **Business Impact** (1-10), **Implementation Feasibility** (1-10), và **Risk Profile**.

---

### REC-01: Chương trình ưu tiên First-Time Buyer

| Dimension | Assessment |
|---|---|
| **Linked Insights** | [INS-02], [INS-04] |
| **Action** | Đánh dấu First-Time Buyer trong OMS, ưu tiên fulfillment và giao hàng đúng hạn tuyệt đối cho nhóm này |
| **Business Impact** | **9/10** — Bảo vệ first-impression, tăng RPR dài hạn |
| **Feasibility** | **8/10** — Chỉ cần flag trong OMS + SLA ưu tiên với carrier |
| **Risk Profile** | Thấp — Không tốn thêm chi phí đáng kể |
| **Priority Tier** | **P1 — Triển khai ngay** |

---

### REC-02: Cảnh báo tự động + Voucher đền bù khi phát hiện delay

| Dimension | Assessment |
|---|---|
| **Linked Insights** | [INS-03], [INS-04] |
| **Action** | Khi hệ thống phát hiện đơn hàng sẽ trễ (dựa trên tracking carrier), tự động gửi thông báo xin lỗi + voucher 10-15% cho đơn tiếp theo |
| **Business Impact** | **8/10** — Giảm tỷ lệ 1-2 sao, bảo vệ seller rating trên marketplace |
| **Feasibility** | **7/10** — Cần tích hợp tracking API + logic trigger |
| **Risk Profile** | Trung bình — Chi phí voucher cần được kiểm soát |
| **Priority Tier** | **P1 — Triển khai ngay** |

---

### REC-03: Tối ưu logistics vùng Đông Bắc

| Dimension | Assessment |
|---|---|
| **Linked Insights** | [INS-05] |
| **Action** | Hợp tác với ≥2 đơn vị vận chuyển tư nhân nội tỉnh tại AL, MA, PI, CE; thiết lập kho trung chuyển gần khu vực tiêu thụ |
| **Business Impact** | **7/10** — Giảm late rate vùng Đông Bắc từ ~20% xuống <10% |
| **Feasibility** | **5/10** — Cần đàm phán partner logistics + vốn đầu tư kho |
| **Risk Profile** | Cao — ROI phụ thuộc vào volume đơn hàng vùng xa |
| **Priority Tier** | **P2 — Triển khai trong 3-6 tháng** |

---

### REC-04: Loyalty program + Email marketing cá nhân hóa

| Dimension | Assessment |
|---|---|
| **Linked Insights** | [INS-01], [INS-06] |
| **Action** | Gửi voucher giảm giá cho đơn hàng thứ 2 trong vòng 30 ngày sau khi đơn đầu giao thành công. Ưu đãi trả góp 0% cho 3 kỳ đầu |
| **Business Impact** | **8/10** — Tăng RPR từ ~3% lên mục tiêu 5-7% |
| **Feasibility** | **7/10** — Cần CRM system + email automation |
| **Risk Profile** | Thấp — Chi phí voucher có thể kiểm soát theo biên lợi nhuận |
| **Priority Tier** | **P1 — Triển khai ngay** |


## 8. Giới hạn Dữ liệu & Lưu ý Phân tích (Data Limitations)

Báo cáo này cần được đọc với nhận thức về các giới hạn sau:

1. **Thiếu dữ liệu chi phí (COGS):** Dataset không chứa giá vốn hàng bán, do đó các phân tích revenue chỉ phản ánh doanh thu/GMV chứ không phản ánh lợi nhuận thực tế. Recommendations liên quan đến voucher/subsidy cần được đánh giá thêm dựa trên biên lợi nhuận thực.

2. **Phạm vi thời gian hạn chế (2016-2018):** Dữ liệu chỉ bao phủ ~2 năm hoạt động, giai đoạn Olist còn đang tăng trưởng nhanh. RPR thấp có thể một phần do thời gian quan sát chưa đủ dài để nhiều khách hàng quay lại.

3. **Mô hình Marketplace vs Direct:** Olist là marketplace aggregator — khách hàng có thể không nhận ra mình đang mua qua Olist, dẫn đến brand loyalty thấp tự nhiên. RPR của Olist không nên so sánh trực tiếp với các D2C e-commerce.

4. **Late rate methodology:** Pipeline sử dụng **full datetime comparison** (delivery_timestamp > estimated_timestamp), cho kết quả late rate = 8.11%. Nếu sử dụng **date-only comparison** (chỉ so sánh ngày, bỏ qua giờ), kết quả sẽ thấp hơn (~6.77%). Cả hai phương pháp đều hợp lệ; report này ghi nhận cả hai và giải thích rõ methodology.
